<a href="https://colab.research.google.com/github/naman-0804/learning/blob/Langchain/genai_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!pip install -q -U langchain langchain-community langchain-google-genai langchain-chroma pypdf fpdf python-dotenv

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━

In [4]:
from fpdf import FPDF

# Create a dummy PDF for testing
pdf = FPDF()
pdf.add_page()
pdf.set_font('Arial', 'B', 16)
pdf.cell(200, 10, 'Machine Learning Overview', ln=True, align='C')
pdf.ln(10)
pdf.set_font('Arial', '', 12)

text = (
    'Supervised learning is a machine learning approach where models learn from labeled training data. '
    'It involves mapping input data to known output labels to make predictions. '
    'Common examples include linear regression, decision trees, and neural networks.\n\n'
    'Unsupervised learning, on the other hand, deals with unlabeled data and tries to find hidden patterns. '
    'Clustering and dimensionality reduction are typical tasks in unsupervised learning.\n\n'
    'Retrieval-Augmented Generation (RAG) is a technique that combines retrieval of relevant documents '
    'with generative models like Gemini to provide accurate answers based on specific data.'
)

for paragraph in text.split('\n\n'):
    pdf.multi_cell(0, 10, paragraph)
    pdf.ln(5)

pdf.output('document.pdf')
print('Generated document.pdf for testing.')

Generated document.pdf for testing.


In [14]:
import os
from google.colab import userdata

try:
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    from langchain_chroma import Chroma
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
except ImportError:
    print('Dependencies missing. Please run the installation cell first.')

# Securely retrieve the API key from Colab Secrets
# Checking both GEMINI_API_KEY and GOOGLE_API_KEY for convenience
try:
    try:
        api_key = userdata.get('GEMINI_API_KEY')
    except userdata.SecretNotFoundError:
        api_key = userdata.get('GEMINI_API_KEY')

    os.environ['GEMINI_API_KEY'] = api_key
except userdata.SecretNotFoundError:
    raise ValueError("Please add 'GEMINI_API_KEY' to the Secrets tab (key icon) and enable notebook access.")

# 1. Load PDF
loader = PyPDFLoader('document.pdf')
documents = loader.load()

# 2. Split document
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# 3. Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

# 4. Store embeddings in ChromaDB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='my_documents',
    persist_directory='./chroma_db'
)

# 5. Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# 6. Gemini LLM
llm = ChatGoogleGenerativeAI(model='gemini-3.7-flash', temperature=0)

# 7. Prompt
prompt = ChatPromptTemplate.from_template("""
You are a document question-answering assistant.
Answer the question using ONLY the provided context.
If the answer is not present in the context, say that you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# 9. RAG Chain
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
)

print('RAG Chain ready.')
response = rag_chain.invoke('What is supervised learning?')
print('\nAnswer:', response.content)

RAG Chain ready.

Answer: [{'type': 'text', 'text': 'Based on the provided context, supervised learning is a machine learning approach where models learn from labeled training data. It involves mapping input data to known output labels to make predictions. Common examples include linear regression, decision trees, and neural networks.', 'extras': {'signature': 'Et0GCtoGARFNMg8MvCbQ0XH8NCE7IRVfW7Hu2LAXopJQYPhweXDQvn+vNYFzC1yDiOe8iJ1PTv9yP//uGGxJUUIpPkSt50PsnC0G8t84m31O6tL+QHRpiP7+74A4UJbMBHnVlecBqgSNIltCZlAgmdjigxE2udsJmViBHi9fVTBlf4daRHQ+e9b+3jFAm5W98IiHTFKsKZGzm55BnfQt6DB5OQObWu9ULHIcOhT9zgrUmXQ87zGcmwUuakSzlcVu28Jj1QG5aVxdqJUpArue1SVzS3Lqlgu3EcmEmgSX0USTVerWNj+WgbZRZkHOfoA+aZ5RYvWxUt0MvXyzEvGhKYE2kyFm2Li4MhQNk4jpyxrq3P7O1tmAbWWB5hmkBq3dU2t6tSV9r5YH9n0jWGean5Zf55vt3aIUVb2xFZ7xrM1MD8Pj1SbyqtLDRW9mnTNKAJAr4K+5xK7MC6mOk/4TV3fnSYiUCeLLWQqbtJ4ggDgsCvdZKCOkyVKjLfuCeqSt0/WoWcaNFRWg/X6Q/eZc+xqFN/Da2e5x9oiCTZ8cInxArpepNbBs/5DQYb9b+m4Ex7J5F8kAq6jWHkCtlIfAbBd6GuquTlA/9qPYBk1W3ZKN6EXDOQ845hGHHVgO

# Gemini RAG Application — Complete Code Explanation

This project builds a basic **RAG (Retrieval-Augmented Generation)** application using:

* LangChain
* Gemini
* Gemini Embeddings
* ChromaDB
* PDF documents

The application allows a user to ask questions about a PDF. Instead of sending the entire PDF to Gemini, it retrieves only the relevant pieces and gives those pieces to Gemini as `{context}`.

---

# 1. Import Required Libraries

```python
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
```

## `os`

```python
import os
```

Provides Python functions for interacting with the operating system.

In this project, it is not strictly necessary because the API key can be loaded through `dotenv`, but it is commonly used when working with environment variables.

---

## `load_dotenv`

```python
from dotenv import load_dotenv
```

Loads variables from a `.env` file into the environment.

For example:

```text
GOOGLE_API_KEY=abc123
```

Then Python can access that environment variable.

---

## `PyPDFLoader`

```python
from langchain_community.document_loaders import PyPDFLoader
```

Loads a PDF and converts its contents into LangChain `Document` objects.

Example:

```text
document.pdf
     ↓
PyPDFLoader
     ↓
Page 1 → Document
Page 2 → Document
Page 3 → Document
```

---

## `RecursiveCharacterTextSplitter`

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter
```

Splits large documents into smaller chunks.

This is necessary because embedding and retrieval work better when the document is divided into manageable pieces.

---

## `ChatGoogleGenerativeAI`

```python
from langchain_google_genai import ChatGoogleGenerativeAI
```

Connects LangChain to Google's Gemini chat model.

It is responsible for generating the final answer.

---

## `GoogleGenerativeAIEmbeddings`

```python
from langchain_google_genai import GoogleGenerativeAIEmbeddings
```

Converts text into numerical vectors called **embeddings**.

Example:

```text
"Python is a programming language"
                 ↓
          Embedding Model
                 ↓
[0.12, -0.45, 0.73, ...]
```

These vectors allow ChromaDB to perform semantic similarity searches.

---

## `Chroma`

```python
from langchain_chroma import Chroma
```

Chroma is the vector database used in this project.

It stores:

```text
Document chunk
      +
Embedding
      +
Metadata
```

and allows us to retrieve chunks that are semantically similar to a user's question.

---

## `ChatPromptTemplate`

```python
from langchain_core.prompts import ChatPromptTemplate
```

Creates a structured prompt for Gemini.

It allows variables such as:

```text
{context}
{question}
```

to be inserted dynamically.

---

## `RunnablePassthrough`

```python
from langchain_core.runnables import RunnablePassthrough
```

Passes an input through the LangChain chain without modifying it.

Here, it is used to pass the user's question directly into:

```text
{question}
```

---

# 2. Load Environment Variables

```python
load_dotenv()
```

This reads the `.env` file.

Example `.env`:

```text
GOOGLE_API_KEY=your_api_key
```

After:

```python
load_dotenv()
```

the Gemini integration can access the API key.

---

# 3. Load the PDF

```python
loader = PyPDFLoader("document.pdf")

documents = loader.load()

print(f"Loaded {len(documents)} pages")
```

## Step 1

```python
loader = PyPDFLoader("document.pdf")
```

Creates a PDF loader.

The PDF file is:

```text
document.pdf
```

---

## Step 2

```python
documents = loader.load()
```

Actually loads the PDF.

The result is a list of LangChain `Document` objects.

Conceptually:

```text
documents = [
    Document(page 1),
    Document(page 2),
    Document(page 3),
    ...
]
```

Each document contains information such as:

```text
page_content
metadata
```

---

## Step 3

```python
print(f"Loaded {len(documents)} pages")
```

Counts how many `Document` objects were created.

For a 20-page PDF:

```text
Loaded 20 pages
```

---

# 4. Split the Document

```python
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")
```

A PDF page may contain too much information to treat as one retrieval unit.

Therefore we split it into smaller chunks.

---

## `chunk_size`

```python
chunk_size=1000
```

Attempts to make each chunk approximately 1000 characters.

For example:

```text
Original document:

[5000 characters]
```

becomes approximately:

```text
Chunk 1 → 1000 characters
Chunk 2 → 1000 characters
Chunk 3 → 1000 characters
Chunk 4 → 1000 characters
Chunk 5 → 1000 characters
```

---

## `chunk_overlap`

```python
chunk_overlap=200
```

Means adjacent chunks share approximately 200 characters.

For example:

```text
Chunk 1:
AAAAAAAAAAAAAAAA
        ↓
       200 characters overlap
        ↓
Chunk 2:
        AAAAAAAAAAAAAAAA
```

Overlap prevents important information from being lost when a sentence or concept lies near a chunk boundary.

---

## Create chunks

```python
chunks = text_splitter.split_documents(documents)
```

The original documents are converted into smaller documents.

The architecture becomes:

```text
PDF
 ↓
Pages
 ↓
Documents
 ↓
Chunks
```

---

# 5. Create Gemini Embeddings

```python
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)
```

This creates the embedding model.

Its job is **not to answer questions**.

Its job is to convert text into vectors.

For example:

```text
"Machine learning uses training data"
                    ↓
             Gemini Embeddings
                    ↓
        [0.21, -0.53, 0.18, ...]
```

The same process happens when the user's question is converted into a vector.

This allows semantic similarity to be calculated.

---

# 6. Store Chunks in ChromaDB

```python
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_documents",
    persist_directory="./chroma_db"
)
```

This is where the chunks and their embeddings are stored.

---

## `documents`

```python
documents=chunks
```

These are the text chunks created earlier.

---

## `embedding`

```python
embedding=embeddings
```

Tells Chroma which embedding model to use.

Each chunk becomes something like:

```text
Chunk
  ↓
Gemini Embedding
  ↓
Vector
  ↓
ChromaDB
```

---

## `collection_name`

```python
collection_name="my_documents"
```

Gives the Chroma collection a name.

Think of a collection as similar to a collection/table containing related vectors.

---

## `persist_directory`

```python
persist_directory="./chroma_db"
```

Specifies where Chroma stores its local database.

The project can therefore have:

```text
project/
│
├── app.py
├── document.pdf
├── .env
└── chroma_db/
```

---

# 7. Create the Retriever

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)
```

This converts the vector database into a **retriever**.

The retriever's job is:

```text
User Question
      ↓
Convert question to embedding
      ↓
Search ChromaDB
      ↓
Find similar chunks
      ↓
Return relevant chunks
```

---

## `k=4`

```python
search_kwargs={"k": 4}
```

Means retrieve approximately the top 4 relevant chunks.

For example:

```text
Question:
"What is supervised learning?"

       ↓

ChromaDB search

       ↓

Chunk 17
Chunk 43
Chunk 51
Chunk 78
```

Those chunks become the context given to Gemini.

---

# 8. Initialize Gemini

```python
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)
```

This creates the actual **LLM**.

Unlike the embedding model, this model generates natural-language answers.

---

## `model`

```python
model="gemini-2.5-flash"
```

Specifies which Gemini model should generate the response.

---

## `temperature`

```python
temperature=0
```

Controls randomness.

A lower temperature makes the output more deterministic.

For a document Q&A system, a low temperature is useful because we generally want consistent answers based on the retrieved context.

---

# 9. Create the Prompt

```python
prompt = ChatPromptTemplate.from_template("""
You are a document question-answering assistant.

Answer the question using ONLY the provided context.

If the answer is not present in the context,
say that you don't know.

Context:
{context}

Question:
{question}

Answer:
""")
```

This defines the instructions given to Gemini.

The two important variables are:

```text
{context}
{question}
```

---

# 10. Understanding `{context}`

This is the most important part of the RAG system.

Suppose the user asks:

```text
What is supervised learning?
```

The retriever finds:

```text
Supervised learning is a machine learning approach
where models learn from labeled training data...
```

That retrieved text is inserted into:

```text
{context}
```

So the final prompt becomes approximately:

```text
You are a document question-answering assistant.

Answer the question using ONLY the provided context.

Context:

Supervised learning is a machine learning approach
where models learn from labeled training data...

Question:

What is supervised learning?

Answer:
```

Gemini then generates the answer.

Therefore:

```text
{context}
```

is not the database itself.

It is the **retrieved text from the database** that is inserted into the prompt.

---

# 11. Format Retrieved Documents

```python
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )
```

The retriever returns multiple `Document` objects.

For example:

```python
[
    Document(page_content="Chunk 1..."),
    Document(page_content="Chunk 2..."),
    Document(page_content="Chunk 3...")
]
```

Gemini does not need the Python `Document` objects themselves.

We extract their text:

```python
doc.page_content
```

and combine them.

Result:

```text
Chunk 1...

Chunk 2...

Chunk 3...
```

This combined text becomes `{context}`.

---

# 12. Build the RAG Chain

```python
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)
```

This is the core LangChain pipeline.

Read it from left to right.

```text
User Question
      │
      ├──────────────→ Retriever → format_docs
      │                       │
      │                       ↓
      │                    context
      │
      └──────────────→ RunnablePassthrough
                              │
                              ↓
                           question

              ↓

           Prompt

              ↓

           Gemini

              ↓

          Final Answer
```

---

# 13. Understanding This Part

```python
"context": retriever | format_docs
```

The user's question goes into the retriever.

The retriever finds relevant documents.

Then:

```python
format_docs
```

converts those documents into plain text.

The resulting text is assigned to:

```python
"context"
```

Therefore:

```text
question
   ↓
retriever
   ↓
relevant documents
   ↓
format_docs
   ↓
context
```

---

# 14. Understanding `RunnablePassthrough`

```python
"question": RunnablePassthrough()
```

The user's original question is passed directly through.

For example:

```text
User:
"What is supervised learning?"
```

becomes:

```python
{
    "context": "retrieved information...",
    "question": "What is supervised learning?"
}
```

This matches the prompt:

```text
Context:
{context}

Question:
{question}
```

---

# 15. Prompt → Gemini

```python
| prompt
| llm
```

The dictionary containing:

```python
{
    "context": "...",
    "question": "..."
}
```

is passed into the prompt.

The prompt produces the complete instruction.

Then:

```python
llm
```

sends that prompt to Gemini.

So:

```text
Retriever output
       +
User question
       ↓
Prompt
       ↓
Gemini
       ↓
Answer
```

---

# 16. Ask Questions

```python
while True:

    question = input("\nAsk a question (or type exit): ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(question)

    print("\nAnswer:")
    print(response.content)
```

This creates a simple command-line chatbot.

---

## Take user input

```python
question = input("\nAsk a question (or type exit): ")
```

Example:

```text
Ask a question:
What is supervised learning?
```

---

## Exit condition

```python
if question.lower() == "exit":
    break
```

If the user enters:

```text
exit
```

the loop stops.

---

# 17. Execute the RAG Pipeline

```python
response = rag_chain.invoke(question)
```

This single line triggers the entire RAG pipeline.

Conceptually:

```text
"What is supervised learning?"
              ↓
          Retriever
              ↓
      Search ChromaDB
              ↓
       Top 4 chunks
              ↓
        format_docs()
              ↓
           context
              ↓
      Construct Prompt
              ↓
           Gemini
              ↓
          Response
```

---

# 18. Print the Answer

```python
print(response.content)
```

The Gemini response is a message object.

Its actual generated text is accessed through:

```python
response.content
```

Example:

```text
Answer:

Supervised learning is a machine learning approach
where a model learns from labeled examples.
```

---

# Complete Architecture

The entire application can be understood as two separate pipelines.

## A. Data Ingestion Pipeline

This happens when the application processes the PDF:

```text
PDF
 ↓
PyPDFLoader
 ↓
Documents
 ↓
Text Splitter
 ↓
Chunks
 ↓
Gemini Embeddings
 ↓
Vectors
 ↓
ChromaDB
```

## B. Question-Answering Pipeline

This happens every time the user asks a question:

```text
User Question
      ↓
Retriever
      ↓
ChromaDB
      ↓
Relevant Chunks
      ↓
format_docs()
      ↓
{context}
      +
{question}
      ↓
Prompt
      ↓
Gemini
      ↓
Answer
```

---

# Why This Is RAG

Without RAG:

```text
User
 ↓
Gemini
 ↓
Answer
```

Gemini relies primarily on its existing model knowledge.

With RAG:

```text
User
 ↓
Retriever
 ↓
Your own documents
 ↓
Relevant context
 ↓
Gemini
 ↓
Answer
```

The model gets external information retrieved from your own data before generating the answer.

---

# The Most Important Components

```text
PyPDFLoader
    ↓
Reads the data

TextSplitter
    ↓
Breaks data into chunks

Gemini Embeddings
    ↓
Converts chunks into vectors

ChromaDB
    ↓
Stores vectors

Retriever
    ↓
Finds relevant chunks

{context}
    ↓
Carries retrieved information into the prompt

Gemini
    ↓
Generates the final answer
```

The central concept to understand from this project is:

```text
Retriever → {context} → Prompt → Gemini
```

That is the fundamental mechanism behind a basic LangChain RAG application.
